# Hyperparameter Tuning (Ray Tune)

<div style="text-align: justify">

The following notebook is dedicated to <b>hyperparameter optimisation</b> for the <b>Tau Anomaly Detection</b> analysis. Using <b>Ray Tune</b> with the <b>ASHA scheduler</b> for aggressive early stopping, a search is performed over the hyperparameter space for the autoencoder (AE) or variational autoencoder (VAE). Models are trained on <b>background only</b> using PyTorch Lightning, and the best configuration is exported as a Hydra-compatible YAML config.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis, model, and tuning configuration |
| Load | `io.load_dataframe` | Read mc.parquet from feature engineering output |
| Search Space | `tuning.build_search_space` | Build Ray Tune search space from config |
| Tune | `tuning.run_tune` | Run Ray Tune with ASHA scheduler and Lightning training |
| Results | `ray.tune` | Best trial summary and trial dataframe |
| Export | `tuning.export_best_config` | Save best config as Hydra-compatible YAML |

The same pipeline is available as a CLI via `uv run python run.py stage=tune`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Data Processing:
* [Pandas](https://pandas.pydata.org/)

Hyperparameter Optimisation:
* [Ray Tune](https://docs.ray.io/en/latest/tune/index.html)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
* [scikit-learn](https://scikit-learn.org/stable/)

### Notebook

Activating autoreload of imported modules.

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [2]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [3]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration. All analysis parameters (run, region, channel, model hyperparameters, tuning settings) are defined in `configs/` and can be overridden here.

> **Model selection:** change `overrides` to `["model=vae"]` for VAE tuning.

In [4]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=ae"])

Resolving input and output directories from config.

In [5]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]

models_dir.mkdir(parents=True, exist_ok=True)

## Data Loading & Preparation

Loading the MC DataFrame produced by the feature engineering pipeline.

In [6]:
from src.processing.io import load_dataframe

df_mc = load_dataframe(dataframes_dir / "mc.parquet")
print(f"Loaded: {len(df_mc):,} events, {len(df_mc.columns)} columns")

Loaded: 9,827,169 events, 56 columns


## Search Space

Building the Ray Tune search space from the tuning config. The search space is defined in `configs/tuning/default.yaml` with separate entries for AE and VAE.

Separating training features and event weights from the MC DataFrame. For anomaly detection, models train on **background only** — signal events are held out for evaluation.

In [7]:
from src.models.tuning import build_search_space

model_type = cfg.model.name
search_space = build_search_space(cfg.tuning, model_type)
print(f"Model type: {model_type}")
print(f"Search space parameters: {list(search_space.keys())}")

Model type: ae
Search space parameters: ['n_layers', 'layer_size', 'latent_dim', 'dropout', 'learning_rate', 'weight_decay', 'batch_size']


## Ray Tune Search

Running Ray Tune with the ASHA scheduler. Each trial trains an AE/VAE on the background-only train split using PyTorch Lightning and reports `val_loss` for early stopping.

In [8]:
from src.models.tuning import run_tune
from src.pipelines.tune import _get_background_origins

# Build DataModule kwargs (same as the pipeline)
mc_path = str(dataframes_dir / "mc.parquet")
background_origins = _get_background_origins(cfg)

dm_kwargs = {
    "mc_path": mc_path,
    "background_origins": background_origins,
    "normalization": cfg.model.normalization,
    "val_fraction": cfg.pipeline.val_fraction,
    "batch_size": cfg.model.batch_size,
    "seed": cfg.seed,
}

# Reduce for faster iteration (default: 100 samples, 200 epochs)
cfg.tuning.num_samples = 2
cfg.model.n_epochs = 11

# ray.init is handled inside run_tune with proper runtime_env
best_config, trial_df = run_tune(cfg=cfg, dm_kwargs=dm_kwargs)
print("Tuning complete")

2026-03-25 14:39:03,136	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/islazyk/ray_results/tau-anomaly-detection-tuning' in 0.0086s.
2026-03-25 14:39:03,139	INFO tune.py:1041 -- Total run time: 1366.16 seconds (1366.10 seconds for the tuning loop).


Tuning complete


## Results

### Best Trial Summary

In [9]:
print("Best config:")
for k, v in best_config.items():
    print(f"  {k}: {v}")

Best config:
  n_layers: 2
  layer_size: 128
  latent_dim: 4
  dropout: 0.0696077235760934
  learning_rate: 7.950259215900751e-05
  weight_decay: 3.3735807163192785e-05
  batch_size: 2048


### Trial Dataframe

Inspecting all trials as a DataFrame for further analysis.

In [10]:
display_cols = [
    "trial_id", "val_loss", "training_iteration",
    "config/n_layers", "config/layer_size", "config/latent_dim",
    "config/dropout", "config/learning_rate", "config/weight_decay",
    "config/batch_size",
]
cols = [c for c in display_cols if c in trial_df.columns]
trial_df[cols].sort_values("val_loss")

,trial_id,val_loss,training_iteration,config/n_layers,config/layer_size,config/latent_dim,config/dropout,config/learning_rate,config/weight_decay,config/batch_size
1,ce5a6_00001,0.014758,11,2,128,4,0.069608,0.000080,0.000034,2048
0,ce5a6_00000,0.023067,11,2,64,4,0.421188,0.007923,0.000033,2048


### Tuning Analysis Plots

Visualizing the hyperparameter search results: optimization history, hyperparameter importance (Spearman correlation with `val_loss`), parallel coordinates, and per-HP scatter plots.

In [ ]:
from src.models.plots import (
    plot_optimization_history,
    plot_hyperparameter_importance,
    plot_parallel_coordinates,
    plot_hp_vs_objective,
)
from src.visualization.plots import save_figure

plots_dir = path / output_paths["plots_dir"] / f"{model_type}_tuning"
plots_dir.mkdir(parents=True, exist_ok=True)

fig = plot_optimization_history(trial_df)
save_figure(fig, plots_dir / "optimization_history.png")

fig = plot_hyperparameter_importance(trial_df)
save_figure(fig, plots_dir / "hp_importance.png")

fig = plot_parallel_coordinates(trial_df)
save_figure(fig, plots_dir / "parallel_coordinates.png")

fig = plot_hp_vs_objective(trial_df)
save_figure(fig, plots_dir / "hp_vs_objective.png")

print(f"Tuning plots saved to: {plots_dir.relative_to(path)}")

## Export Best Parameters

Exporting the best trial's configuration as a Hydra-compatible YAML model config. This file can be copied to `configs/model/` and used for training:

```bash
cp <output_path> configs/model/ae_tuned.yaml
uv run python run.py stage=train model=ae_tuned
```

In [12]:
import json
import ray
from src.models.tuning import export_best_config                                                                                                                                                                   

best_model_config = export_best_config(best_config, model_type)                                                                                                                                                    
                                                                                                                                                                                                                
params_path = models_dir / f"{model_type}_best_tuning.json"
with open(params_path, "w") as f:
    json.dump(best_model_config, f, indent=2)

print(f"Best params exported to: {params_path.relative_to(path)}")
print(json.dumps(best_model_config, indent=2))

ray.shutdown()

Best params exported to: data/processed/ML/25.2.28/Run2/Preselection/1_tau/models/ae_best_tuning.json
{
  "hidden_sizes": [
    128,
    64
  ],
  "latent_dim": 4,
  "dropout": 0.069608,
  "learning_rate": 7.950259215900751e-05,
  "weight_decay": 3.3735807163192785e-05,
  "batch_size": 2048
}
